# BEE 4750 Lab 2: Linear Programming with JuMP

**Name**: Mikael M Rendon

**ID**: mm3446

> **Due Date**
>
> Tuesday, October 27, 9:00pm

## Setup

Run these two cells first. The first time, installing and loading `JuMP`
and `HiGHS` can take a few minutes.

In [6]:
import Pkg
Pkg.activate(".")
Pkg.instantiate()

  Activating project at `~/BEE 4750/lab02`


In [9]:
Pkg.add("JuMP")
Pkg.add("HiGHS")
using JuMP    # optimization modeling syntax
using HiGHS   # the solver

   Resolving package versions...
  No Changes to `~/BEE 4750/lab02/Project.toml`
  No Changes to `~/BEE 4750/lab02/Manifest.toml`
   Resolving package versions...
   Installed HiGHS ────────── v1.26.1
   Installed MathOptIIS ───── v0.2.1
   Installed OpenBLAS32_jll ─ v0.3.34+1
    Updating `~/BEE 4750/lab02/Project.toml`
  [87dc4568] + HiGHS v1.26.1
    Updating `~/BEE 4750/lab02/Manifest.toml`
  [87dc4568] + HiGHS v1.26.1
  [8c4f8055] + MathOptIIS v0.2.1
  [8fd58aa0] + HiGHS_jll v1.15.1+3
  [656ef2d0] + OpenBLAS32_jll v0.3.34+1
Precompiling project...
   2215.7 ms  ✓ HiGHS_jll
   2541.6 ms  ✓ OpenBLAS32_jll
   3596.1 ms  ✓ MathOptIIS
  27366.1 ms  ✓ HiGHS
  4 dependencies successfully precompiled in 32 seconds. 200 already precompiled.


## Introduction

In the October 19 lecture, we saw how to formulate and solve a two-stack
emissions problem in `JuMP` and read its shadow prices. In this lab you
will do the same for three power plants on a hot summer afternoon: write
its linear program in `JuMP`, solve it, read and interpret the shadow
prices, and re-solve it when one limit tightens.

### The Plants

A utility owns three power plants side by side on a river: a coal steam
plant, a combined-cycle plant and a combustion-turbine plant. It sells
all the electricity they generate at the wholesale price, \$120/MWh this
afternoon, and decides how much each plant should generate this hour to
make the most profit. Each plant’s profit per MWh is the price minus its
variable cost (mostly fuel).

| Plant | Capacity (MW) | Variable cost (\$/MWh) | Heat to river (MWth per MW) | NOx (lb/MWh) |
|:--------------|--------------:|--------------:|--------------:|--------------:|
| Coal steam | 600 | 30 | 1.5 | 1 |
| Combined cycle | 400 | 56 | 0.64 | 0.1 |
| Combustion turbines | 300 | 87 | 0 | 0.5 |

The three plants share three limits:

- **River heat.** The coal plant and the combined-cycle plant are cooled
  with river water, which goes back into the river warmer. On a hot day
  the river is already warm, so the state caps the heat the three plants
  together may discharge to this stretch of the river: 1,000 MWth this
  hour. (MWth is megawatts of heat, to keep it apart from the MW of
  electricity the plants generate.) The combustion turbines are
  air-cooled and send no heat to the river.
- **NOx.** Hot, sunny days are also ozone days, and NOx is one of the
  pollutants that forms ozone. Together, the three plants may emit at
  most 650 lb of NOx this hour.
- **Transmission line.** The plants share one line to the grid, which
  can carry at most 1,000 MW.

The same data in code:

In [26]:
plant_names = ["Coal steam", "Combined cycle", "Combustion turbines"]
n_plants = length(plant_names)

price = 120                     # electricity price           [$/MWh]
variable_cost = [30, 56, 87]    # fuel and operating cost     [$/MWh]
capacity = [600, 400, 300]      # maximum output              [MW]
heat_to_river = [1.5, 0.64, 0]  # heat sent to the river      [MWth per MW]
nox_rate = [1, 0.1, 0.5]        # NOx emitted                 [lb/MWh]

heat_limit = 1000               # river heat permit           [MWth]
nox_limit = 650                 # NOx limit                   [lb]
line_limit = 1000               # transmission line capacity  [MW]

1000

> **Two Ways to Work Through This Lab**
>
> You can do the coding either in this notebook (`lab02.ipynb`) or in
> the plain script `lab02.jl`, whichever you prefer: both contain the
> same problems. Either way, the written parts go on the separate
> **Activity Sheet** (handed out in class). Submit one PDF with your
> code output and your completed Activity Sheet to Gradescope by 9:00pm
> on Tuesday, October 27.

> **JuMP Syntax**
>
> From the October 19 lecture, a `JuMP` model has three kinds of pieces
> after the model itself:
>
> ``` julia
> model = Model(HiGHS.Optimizer)
> @variable(model, 0 <= amount[i in 1:n_items] <= upper_bound[i])
> @objective(model, Max, sum(unit_value[i] * amount[i] for i in 1:n_items))
> @constraint(model, budget,
>     sum(unit_cost[i] * amount[i] for i in 1:n_items) <= available)
> ```
>
> Use `Max` or `Min` as the problem needs. The name in `@constraint`
> (here `budget`) is how you ask about that constraint later. After
> `optimize!(model)`:
>
> - `value(amount[1])`, or `value.(amount)` for all of them, gives the
>   optimal decisions;
> - `objective_value(model)` gives the optimal objective;
> - `value(budget)` gives the constraint’s left-hand side at the
>   optimum;
> - `shadow_price(budget)` gives the change in the objective when
>   `budget` is relaxed by one unit.
>
> Put the whole model in one cell and re-run that whole cell after any
> change. `JuMP` will not add a second constraint with the same name to
> a model, so re-running a single `@constraint` line gives an error.

## Problem 1: Formulate the LP

Before writing any code, complete Problem 1 on the Activity Sheet: write
the utility’s objective and the three shared limits as a linear program,
with units. Use $y_1$, $y_2$ and $y_3$ for the output of the coal plant,
the combined-cycle plant and the combustion-turbine plant this hour, in
MW. Each is at least 0 and at most its plant’s capacity.

## Problem 2: Solve It in JuMP

Complete the model below from your Problem 1 formulation. The capacities
are already in the `@variable` bounds. Write each of the three shared
limits as a **named** constraint, using the names in the comments: the
names are what let `JuMP` report their shadow prices in Problem 3.

In [27]:
utility = Model(HiGHS.Optimizer)
set_silent(utility)

# Output of each plant this hour [MW], between 0 and the plant's capacity
@variable(utility, 0 <= generation[i = 1:n_plants] <= capacity[i])

# objective functions
@objective(utility, Max, sum((price-variable_cost[i])*generation[i] for i in 1:n_plants))

# constraints
@constraint(utility, river_heat, sum(heat_to_river[i]*generation[i] for i in 1:n_plants) <= heat_limit)
@constraint(utility, nox_emissions, sum(nox_rate[i]*generation[i] for i in 1:n_plants) <= nox_limit)
@constraint(utility, transmission_line, sum(generation[i] for i in 1:n_plants) <= line_limit)

print(utility)

Max 90 generation[1] + 64 generation[2] + 33 generation[3]
Subject to
 river_heat : 1.5 generation[1] + 0.64 generation[2] ≤ 1000
 nox_emissions : generation[1] + 0.1 generation[2] + 0.5 generation[3] ≤ 650
 transmission_line : generation[1] + generation[2] + generation[3] ≤ 1000
 generation[1] ≥ 0
 generation[2] ≥ 0
 generation[3] ≥ 0
 generation[1] ≤ 600
 generation[2] ≤ 400
 generation[3] ≤ 300


Check the printed model against your Activity Sheet, then solve it. The
status should be `OPTIMAL`.

In [28]:
optimize!(utility)
termination_status(utility)

OPTIMAL::TerminationStatusCode = 1

In [29]:
for i in 1:n_plants
    println(plant_names[i], ": ", value(generation[i]), " MW")
end
println("Profit: \$", objective_value(utility))

Coal steam: 496.0 MW
Combined cycle: 400.0 MW
Combustion turbines: 104.0 MW
Profit: $73672.0


If your model matches the problem, the profit is \$73,672 for the hour.
If you get a different number, compare `print(utility)` with your
Problem 1 formulation, or ask the TA.

How much of each limit does this plan use?

In [30]:
println("River heat: ", value(river_heat), " of ", heat_limit, " MWth")
println("NOx: ", value(nox_emissions), " of ", nox_limit, " lb")
println("Line: ", value(transmission_line), " of ", line_limit, " MW")

River heat: 1000.0 of 1000 MWth
NOx: 588.0 of 650 lb
Line: 1000.0 of 1000 MW


Record the plan in Problem 2 on the Activity Sheet and answer the
question there.

## Problem 3: Shadow Prices

Use `shadow_price` to get the shadow price of each of the three named
constraints.

In [31]:

println("River heat: ", shadow_price(river_heat))
println("NOx: ", shadow_price(nox_emissions))
println("Line: ", shadow_price(transmission_line))  

River heat: 38.0
NOx: 0.0
Line: 33.0


`JuMP`’s `shadow_price` is the change in the objective when a constraint
is relaxed by one unit. This model maximizes profit, and relaxing a
limit can only raise the best profit or leave it unchanged, so expect
shadow prices that are positive or zero. (In the cost-minimizing
emissions problem from lecture they were negative or zero, for the same
reason.)

Complete Problem 3 on the Activity Sheet.

## Problem 4: A Hotter River

The heat wave deepens: the river warms further, and the state cuts the
plants’ heat limit from 1,000 to 850 MWth. Answer Problem 4a on the
Activity Sheet, a prediction from the shadow price, **before** you
re-solve.

First, save the results under the current limit, since re-solving will
replace them:

In [32]:
base_generation = value.(generation)
base_profit = objective_value(utility)
base_nox = value(nox_emissions)

588.0

Now lower the heat limit and solve again. `set_normalized_rhs` changes a
constraint’s right-hand side in the model you already built, as in the
October 19 lecture.

In [33]:
heatwave_heat_limit = 850   # [MWth]

set_normalized_rhs(river_heat, heatwave_heat_limit)
optimize!(utility)

In [37]:
for i in 1:n_plants
    println(plant_names[i], ": ", base_generation[i], " MW before, ",
        value(generation[i]), " MW after")
end
println("Profit: \$", base_profit, " before, \$",
    objective_value(utility), " after")
println("NOx: ", base_nox, " lb before, ", value(nox_emissions), " lb after") 

Coal steam: 496.0 MW before, 396.0 MW after
Combined cycle: 400.0 MW before, 400.0 MW after
Combustion turbines: 104.0 MW before, 204.0 MW after
Profit: $73672.0 before, $67972.0 after
NOx: 588.0 lb before, 538.0 lb after


Complete Problem 4 on the Activity Sheet.

## If You Finish Early (Optional)

How far does the heat limit’s shadow price hold? Re-solve at each heat
limit below and store its shadow price. Where does it change, and why?

In [38]:
heat_limits = [1100, 1000, 900, 800, 700, 600]   # [MWth]
heat_prices = zeros(length(heat_limits))
for k in 1:length(heat_limits)
    # TODO: set the heat limit to heat_limits[k], solve again,
    # and store shadow_price(river_heat) in heat_prices[k]
    set_normalized_rhs(river_heat, heat_limits[k])
    optimize!(utility)
    heat_prices[k] = shadow_price(river_heat)
end
heat_prices

6-element Vector{Float64}:
 38.0
 38.0
 38.0
 38.0
 60.0
 60.0

In this lab the price of \$120/MWh was given. On Wednesday, the economic
dispatch lecture shows where a price like that comes from.